<a href="https://colab.research.google.com/github/GiX007/agent-labs/blob/main/08_dspy/football_player_guessing_game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Football Player Guessing Game 🎮⚽

A DSPy-powered game where the AI tries to guess your favorite football player by asking yes/no questions. Built with Qwen3-8B on HuggingFace.

## Setup

In [1]:
# Install dspy
!pip install -q dspy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.5/146.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 8.8 MB/s eta 0:00:00


In [2]:
# Load HF token
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [3]:
# Configure DSPy with Qwen3.5-9B
import dspy
import warnings
import logging

warnings.filterwarnings("ignore", message=".*max_retries.*")
logging.getLogger("dspy.clients.lm").setLevel(logging.ERROR)
logging.getLogger("litellm").setLevel(logging.ERROR)

lm = dspy.LM("huggingface/Qwen/Qwen3-8B")
dspy.configure(lm=lm)

In [4]:
# Test: does the model respond?
test = lm("Name one famous football player.")
print(test)

[{'text': '\n\nOne famous football (soccer) player is **Lionel Messi**. He is widely regarded as one of the greatest players of all time, known for his exceptional dribbling skills, goal-scoring ability, and success with clubs like Barcelona and Inter Miami, as well as his achievements with the Argentina national team, including winning the FIFA World Cup in 2022.', 'reasoning_content': '\nOkay, the user asked for one famous football player. First, I need to recall some of the most well-known names in football. Let\'s see... There are many, but some of the most iconic ones are Lionel Messi, Cristiano Ronaldo, Erling Haaland, and Kylian Mbappé. These players have won numerous awards and have had significant impacts on the sport.\n\nI should consider their current status. Messi and Ronaldo are still active, though they\'ve moved clubs recently. Haaland is a rising star, and Mbappé is also very prominent. Maybe also include someone from the past, like Diego Maradona or Pele, but the quest

In [5]:
# test is a list, access first element with [0]
print(f"Response text: {test[0]['text']}\n")
print(f"Reasoning text:\n {test[0]['reasoning_content']}")

Response text: 

One famous football (soccer) player is **Lionel Messi**. He is widely regarded as one of the greatest players of all time, known for his exceptional dribbling skills, goal-scoring ability, and success with clubs like Barcelona and Inter Miami, as well as his achievements with the Argentina national team, including winning the FIFA World Cup in 2022.

Reasoning text:
 
Okay, the user asked for one famous football player. First, I need to recall some of the most well-known names in football. Let's see... There are many, but some of the most iconic ones are Lionel Messi, Cristiano Ronaldo, Erling Haaland, and Kylian Mbappé. These players have won numerous awards and have had significant impacts on the sport.

I should consider their current status. Messi and Ronaldo are still active, though they've moved clubs recently. Haaland is a rising star, and Mbappé is also very prominent. Maybe also include someone from the past, like Diego Maradona or Pele, but the question doesn

## Game Logic

In [6]:
# Create a Signature class
class QuestionGenerator(dspy.Signature):
  """Generate a yes/no question to guess the football player the user is thinking of. You can ask general questions or directly guess the name if you have enough clues. Never repeat a question from past_questions."""

  past_questions: list[str] = dspy.InputField(desc="previously asked questions")
  past_answers: list[bool] = dspy.InputField(desc="user answers (True=yes, False=no)")
  new_question: str = dspy.OutputField(desc="next yes/no question to narrow down the player")
  is_guess: bool = dspy.OutputField(desc="True if new_question is a direct name guess, False if still a general question")

In [7]:
# Test: the Signature is just a contract, not yet connected to the LLM
print(QuestionGenerator.__doc__)
print(QuestionGenerator.model_fields.keys())

Generate a yes/no question to guess the football player the user is thinking of. You can ask general questions or directly guess the name if you have enough clues. Never repeat a question from past_questions.
dict_keys(['past_questions', 'past_answers', 'new_question', 'is_guess'])


In [8]:
# Connect the Signature to ChainOfThought so the model reasons before asking
question_generator = dspy.ChainOfThought(QuestionGenerator)

In [9]:
# Test: ask the first question (no history yet)
output = question_generator(past_questions=[], past_answers=[])
print(f"Question: {output.new_question}")
print(f"Is guess: {output.is_guess}")

Question: Is the player a forward?
Is guess: False


In [10]:
# Create the game class
class FootballPlayerGameGuess(dspy.Module):
  """A game where the AI guesses your favorite football player via yes/no questions."""

  def __init__(self, max_questions: int = 15):
    super().__init__()
    self.ask_question = dspy.ChainOfThought(QuestionGenerator)
    self.max_questions = max_questions

  def forward(self) -> None:
    """Run the guessing game loop."""
    print("Think of a football player and I'll try to guess who it is!\n")

    past_questions = []
    past_answers = []

    for i in range(self.max_questions):
      # Generate next question
      result = self.ask_question(past_questions=past_questions, past_answers=past_answers)

      # Get user's yes/no answer
      while True:
        answer = input(f"Q{i+1}: {result.new_question} (y/n): ").strip().lower()
        if answer in ("y", "n"):
          break
        print("Please enter y or n.")

      # Store history
      past_questions.append(result.new_question)
      past_answers.append(answer == "y")

      # Check if the model made a correct guess
      if result.is_guess and answer == "y":
        print("\n🎉 I found it!")
        return

    print("\n😅 I couldn't guess it. You win!")

In [11]:
# Quick test
game = FootballPlayerGameGuess()
game()

Think of a football player and I'll try to guess who it is!

Q1: Is the player a forward? (y/n): y
Q2: Is the player a striker? (y/n): y
Q3: Is the player known for scoring a lot of goals? (y/n): y
Q4: Has the player won the Ballon d'Or? (y/n): n
Q5: Is the player Robert Lewandowski? (y/n): y

🎉 I found it!


## Gradio UI

In [12]:
!pip install -q gradio

In [13]:
import gradio as gr

def start_game():
  """Reset the game state and ask the first question."""
  state = {"past_questions": [], "past_answers": []}
  result = question_generator(past_questions=[], past_answers=[])
  state["pending_question"] = result.new_question
  state["pending_is_guess"] = result.is_guess
  return state, f"Think of a football player!\n\nQ1: {result.new_question}"

def answer(user_answer: str, state: dict):
  """Process yes/no answer and generate next question."""
  is_yes = user_answer == "Yes"

  # Store the answer
  state["past_questions"].append(state["pending_question"])
  state["past_answers"].append(is_yes)

  # Check if last question was a correct guess
  if state["pending_is_guess"] and is_yes:
    return state, f"🎉 I guessed it in {len(state['past_questions'])} questions!"

  # Check if we've hit the limit
  if len(state["past_questions"]) >= 15:
    return state, "😅 I couldn't guess it. You win!"

  # Generate next question
  result = question_generator(past_questions=state["past_questions"], past_answers=state["past_answers"])
  state["pending_question"] = result.new_question
  state["pending_is_guess"] = result.is_guess

  q_num = len(state["past_questions"]) + 1
  return state, f"Q{q_num}: {result.new_question}"

In [14]:
with gr.Blocks(title="Football Player Guessing Game") as demo:
  gr.Markdown("# ⚽ Football Player Guessing Game\nThink of a football player and I'll try to guess who it is!")

  # Game state (hidden from user)
  state = gr.State(value={})

  # Display area for questions
  output_text = gr.Textbox(label="Game", lines=3, interactive=False)

  # Buttons
  with gr.Row():
    yes_btn = gr.Button("Yes", variant="primary")
    no_btn = gr.Button("No", variant="secondary")
  new_game_btn = gr.Button("🎮 New Game")

  # Connect buttons to functions
  new_game_btn.click(fn=start_game, inputs=[], outputs=[state, output_text])
  yes_btn.click(fn=answer, inputs=[gr.State("Yes"), state], outputs=[state, output_text])
  no_btn.click(fn=answer, inputs=[gr.State("No"), state], outputs=[state, output_text])

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7fa783f21ad0d8a27a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Deploy to HuggingFace Spaces

To turn this notebook into a live web app that anyone can use:

1. Go to [huggingface.co/new-space](https://huggingface.co/new-space) and create a new Space (SDK: Gradio, Hardware: CPU Free).

2. Create two files in the Space:
*   **requirements.txt** with just the dependencies: `dspy`.
*   **app.py** with all the game logic combined (Signature, Module, Gradio UI) into one file. Same code from this notebook, without the `input()`-based game class.

3. Add your HuggingFace token as a **Secret** in the Space settings (name: `HF_TOKEN`). This is needed because DSPy calls the HF Inference API to use Qwen3-8B.

4. The Space builds automatically and gives back a permanent public link.

**Our live prototype:** [huggingface.co/spaces/GiX007/football-player-guessing-game](https://huggingface.co/spaces/GiX007/football-player-guessing-game)